In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.stats as stats
pd.options.display.max_columns=None
import warnings

In [2]:
test = pd.read_csv('/kaggle/input/eedi-mining-misconceptions-in-mathematics/test.csv')
test.head()

,QuestionId,ConstructId,ConstructName,SubjectId,SubjectName,CorrectAnswer,QuestionText,AnswerAText,AnswerBText,AnswerCText,AnswerDText
0,1869,856,Use the order of operations to carry out calcu...,33,BIDMAS,A,\[\n3 \times 2+4-5\n\]\nWhere do the brackets ...,\( 3 \times(2+4)-5 \),\( 3 \times 2+(4-5) \),\( 3 \times(2+4-5) \),Does not need brackets
1,1870,1612,Simplify an algebraic fraction by factorising ...,1077,Simplifying Algebraic Fractions,D,"Simplify the following, if possible: \( \frac{...",\( m+1 \),\( m+2 \),\( m-1 \),Does not simplify
2,1871,2774,Calculate the range from a list of data,339,Range and Interquartile Range from a List of Data,B,Tom and Katie are discussing the \( 5 \) plant...,Only\nTom,Only\nKatie,Both Tom and Katie,Neither is correct


In [6]:
train = pd.read_csv('/kaggle/input/eedi-mining-misconceptions-in-mathematics/train.csv')
train.head()

,QuestionId,ConstructId,ConstructName,SubjectId,SubjectName,CorrectAnswer,QuestionText,AnswerAText,AnswerBText,AnswerCText,AnswerDText,MisconceptionAId,MisconceptionBId,MisconceptionCId,MisconceptionDId
0,0,856,Use the order of operations to carry out calcu...,33,BIDMAS,A,\[\n3 \times 2+4-5\n\]\nWhere do the brackets ...,\( 3 \times(2+4)-5 \),\( 3 \times 2+(4-5) \),\( 3 \times(2+4-5) \),Does not need brackets,NaN,NaN,NaN,1672.0
1,1,1612,Simplify an algebraic fraction by factorising ...,1077,Simplifying Algebraic Fractions,D,"Simplify the following, if possible: \( \frac{...",\( m+1 \),\( m+2 \),\( m-1 \),Does not simplify,2142.0,143.0,2142.0,NaN
2,2,2774,Calculate the range from a list of data,339,Range and Interquartile Range from a List of Data,B,Tom and Katie are discussing the \( 5 \) plant...,Only\nTom,Only\nKatie,Both Tom and Katie,Neither is correct,1287.0,NaN,1287.0,1073.0
3,3,2377,Recall and use the intersecting diagonals prop...,88,Properties of Quadrilaterals,C,The angles highlighted on this rectangle with ...,acute,obtuse,\( 90^{\circ} \),Not enough information,1180.0,1180.0,NaN,1180.0
4,4,3387,Substitute positive integer values into formul...,67,Substitution into Formula,A,The equation \( f=3 r^{2}+3 \) is used to find...,\( 30 \),\( 27 \),\( 51 \),\( 24 \),NaN,NaN,NaN,1818.0


In [4]:
df = pd.concat([train,test])
df.head()

,QuestionId,ConstructId,ConstructName,SubjectId,SubjectName,CorrectAnswer,QuestionText,AnswerAText,AnswerBText,AnswerCText,AnswerDText,MisconceptionAId,MisconceptionBId,MisconceptionCId,MisconceptionDId
0,0,856,Use the order of operations to carry out calcu...,33,BIDMAS,A,\[\n3 \times 2+4-5\n\]\nWhere do the brackets ...,\( 3 \times(2+4)-5 \),\( 3 \times 2+(4-5) \),\( 3 \times(2+4-5) \),Does not need brackets,NaN,NaN,NaN,1672.0
1,1,1612,Simplify an algebraic fraction by factorising ...,1077,Simplifying Algebraic Fractions,D,"Simplify the following, if possible: \( \frac{...",\( m+1 \),\( m+2 \),\( m-1 \),Does not simplify,2142.0,143.0,2142.0,NaN
2,2,2774,Calculate the range from a list of data,339,Range and Interquartile Range from a List of Data,B,Tom and Katie are discussing the \( 5 \) plant...,Only\nTom,Only\nKatie,Both Tom and Katie,Neither is correct,1287.0,NaN,1287.0,1073.0
3,3,2377,Recall and use the intersecting diagonals prop...,88,Properties of Quadrilaterals,C,The angles highlighted on this rectangle with ...,acute,obtuse,\( 90^{\circ} \),Not enough information,1180.0,1180.0,NaN,1180.0
4,4,3387,Substitute positive integer values into formul...,67,Substitution into Formula,A,The equation \( f=3 r^{2}+3 \) is used to find...,\( 30 \),\( 27 \),\( 51 \),\( 24 \),NaN,NaN,NaN,1818.0


In [5]:
num_rows,num_cols = df.shape
print(F"Number of rows: {num_rows}")
print(F"Number of columns:{num_cols}")

continuous_vars = df.select_dtypes(include=['float64','int64']).columns
categorical_vars = df.select_dtypes(include=['object','category']).columns

print(F"Continuous variables: {list(continuous_vars)}")
print(F"Categorical variables: {list(categorical_vars)}")



Number of rows: 1872
Number of columns:15
Continuous variables: ['QuestionId', 'ConstructId', 'SubjectId', 'MisconceptionAId', 'MisconceptionBId', 'MisconceptionCId', 'MisconceptionDId']
Categorical variables: ['ConstructName', 'SubjectName', 'CorrectAnswer', 'QuestionText', 'AnswerAText', 'AnswerBText', 'AnswerCText', 'AnswerDText']


In [7]:
numerical_vars = df.select_dtypes(include=['float64','int64'])
five_point_summary = numerical_vars.describe().loc[['min','25%','50%','75%','max']]

print("Five-point summary:")
print(five_point_summary)

Five-point summary:
     QuestionId  ConstructId  SubjectId  MisconceptionAId  MisconceptionBId  \
min        0.00         4.00       33.0               1.0               1.0   
25%      467.75       575.00       92.0             686.0             628.5   
50%      935.50      1470.00      203.0            1336.0            1379.0   
75%     1403.25      2638.25      238.0            1954.0            1970.0   
max     1871.00      3526.00     1984.0            2585.0            2586.0   

     MisconceptionCId  MisconceptionDId  
min              2.00               0.0  
25%            652.25             578.0  
50%           1294.50            1282.0  
75%           1912.00            1897.0  
max           2585.00            2583.0  


In [8]:
categorical_vars = df.select_dtypes(include=['object','category'])

for col in categorical_vars.columns:
    print(F"\nSummary for'{col}':")
    
    num_categories = df[col].nunique()
    print(F"Number of categories:{num_categories}")
    
    category_counts = df[col].value_counts(normalize=True)*100
    print(F"Percentage of observations in each category:\n{category_counts}")
    


Summary for'ConstructName':
Number of categories:757
Percentage of observations in each category:
ConstructName
Calculate the square of a number                                                                                          0.747863
Solve two-step linear equations, with the variable on one side, with all positive integers                                0.694444
Factorise a quadratic expression in the form x² + bx + c                                                                  0.694444
Calculate the range from a list of data                                                                                   0.641026
Use the order of operations to carry out calculations involving addition, subtraction, multiplication, and/or division    0.641026
                                                                                                                            ...   
Shade percentages of a shape where the percent is a multiple of 10                                   

In [10]:
print("Checking for missing/null values:")
missing_values = df.isnull().sum()
print(missing_values[missing_values > 0])

def detect_outliers(column):
    Q1 = np.percentile(column,25)
    Q3 = np.percentile(column,75)
    IQR=Q3-Q1
    lower_bound = Q1-1.5*IQR
    upprer_bound = Q3+1.5*IQR
    return column[(column < lower_bound) | (column > upprer_bound)]

print("\nChecking for outliers:")
numerical_vars = df.select_dtypes(include=['float64','int64'])
for col in numerical_vars.columns:
    outliers = detect_outliers(df[col].dropna())
    if len(outliers) > 0:
        print(F"{col} has {len(outliers)}outliers.")
    
    QuestionId_column = 'QuestionId'
    if QuestionId_column in df.columns:
        print(F"\nClass distribution for '{QuestionId_column}':")
        class_distribution = df[QuestionId_column].value_counts(normalize=True)*100
        print(class_distribution)
        
    imbalance_threshold = 70
    if (class_distribution.max() > imbalance_threshold):
        print(F"\nWarning: class imbalance detected! One class has more than {imbalance_threshold}% of the data.")
    else:
        print(F"\nNo significant class imbalance detected.")
        
        
        
    

Checking for missing/null values:
MisconceptionAId    737
MisconceptionBId    754
MisconceptionCId    792
MisconceptionDId    835
dtype: int64

Checking for outliers:

Class distribution for 'QuestionId':
QuestionId
0       0.053419
1244    0.053419
1256    0.053419
1255    0.053419
1254    0.053419
          ...   
619     0.053419
618     0.053419
617     0.053419
616     0.053419
1871    0.053419
Name: proportion, Length: 1872, dtype: float64

No significant class imbalance detected.

Class distribution for 'QuestionId':
QuestionId
0       0.053419
1244    0.053419
1256    0.053419
1255    0.053419
1254    0.053419
          ...   
619     0.053419
618     0.053419
617     0.053419
616     0.053419
1871    0.053419
Name: proportion, Length: 1872, dtype: float64

No significant class imbalance detected.
SubjectId has 107outliers.

Class distribution for 'QuestionId':
QuestionId
0       0.053419
1244    0.053419
1256    0.053419
1255    0.053419
1254    0.053419
          ...   
619  

In [14]:
from sklearn.utils import resample
from imblearn.over_sampling import SMOTE

def fix_missing_values(df):
    
    for col in df.columns:
        if df[col].isnull().sum() > 0:
            if df[col].dtype == 'object':
                df[col].fillna(df[col].mode()[0], inplace=True)
            else:
                df[col].fillna(df[col].median(), inplace=True)
    return df

df = fix_missing_values(df)
print("Missing values fixed.")

def fix_outliers(df):
    for col in df.select_dtypes(include=['float64','int64']).columns:
        Q1 = np.percentile(df[col],25)
        Q3 = np.percentile(df[col],75)
        IQR = Q3-Q1
        lower_bound = Q1- 1.5*IQR
        upper_bound = Q3+ 1.5*IQR
        
        df[col] = np.where(df[col] < lower_bound,Q1,df[col])
        df[col] = np.where(df[col] > upper_bound,Q3,df[col])
    return df
df = fix_outliers(df)
print("Outliers hanlded.")

QuestionId_column = 'QuestionId'

if QuestionId_column in df.columns:
    print(F"\nOriginal class distribution for'{QuestionId_column}':")
    print(df[QuestionId_column].value_counts())
    
    majority_class = df[df[QuestionId_column] == df[QuestionId_column].mode()[0]]
    minority_class = df[df[QuestionId_column] != df[QuestionId_column].mode()[0]]
    
    minority_upsampled = resample(minority_class,
                                   replace = True,
                                   n_samples=len(majority_class),
                                   random_state=42)
    df_balanced = pd.concat([majority_class, minority_upsampled])
    
    print("\nBlalnced class distribution after oversampling:")
    print(df_balanced[QuestionId_column].value_counts())
    df = df_balanced
print("\ndata cleaning and balancing completed.")
    
        
        
        

Missing values fixed.
Outliers hanlded.

Original class distribution for'QuestionId':
QuestionId
0.0       1
1244.0    1
1256.0    1
1255.0    1
1254.0    1
         ..
619.0     1
618.0     1
617.0     1
616.0     1
1871.0    1
Name: count, Length: 1872, dtype: int64

Blalnced class distribution after oversampling:
QuestionId
0.0       1
1127.0    1
Name: count, dtype: int64

data cleaning and balancing completed.


In [18]:
df.head()

,QuestionId,ConstructId,ConstructName,SubjectId,SubjectName,CorrectAnswer,QuestionText,AnswerAText,AnswerBText,AnswerCText,AnswerDText,MisconceptionAId,MisconceptionBId,MisconceptionCId,MisconceptionDId
0,0.0,856.0,Use the order of operations to carry out calcu...,33.0,BIDMAS,A,\[\n3 \times 2+4-5\n\]\nWhere do the brackets ...,\( 3 \times(2+4)-5 \),\( 3 \times 2+(4-5) \),\( 3 \times(2+4-5) \),Does not need brackets,1336.0,1379.0,1294.5,1672.0
1127,1127.0,1614.0,Simplify an algebraic fraction by factorising ...,238.0,Multiplying and Dividing Algebraic Fractions,B,"Write the following as a single fraction, givi...",\( \frac{4 t^{2}-4 t-48}{4(t-4)} \),\( t+3 \),\( \frac{t^{2}-t-12}{t-4} \),\( t-3 \),1336.0,1379.0,1593.0,1282.0


In [ ]:
QuestionId_column = 'QuestionId'

numerical_vars = df.select_dtypes(include=['float64','int64'])
df[numerical_vars.columns].hist(bins=30, figsize=(15,10), grid=False)
plt.suptitle("Histogram of Numerical variables")
plt.show()

categorical_vars = df.select_dtypes(include=['object','category'])
for col in categorical_vars.columns:
    plt.figure(figsize=(10,5))
    sns.countplot(data=df,x=col)
    plt.title(F"Count plot of {col}")
    plt.xticks(rotation=45)
    plt.show()
    
correlation_matrix = df.corr()

plt.figure(figsize=(12,8))
sns.heatmap(correlation_matrix,annot=True,fmt=".2f", cmap='coolwarm',square=True)
plt.title("Correlation Matrix")
plt.show()

QuestionId_column = correlation_matrix[QuestionId_column].drop(QuestionId_column)
high_correlation_vars = QuestionId_corr[abs(QuestionId_corr) > 0.5]
print("\nVariables highly correlated with the QuestionId:")
print(high_correlation_vars)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

QuestionId_column = 'QuestionId'

variables_to_exclude = ['ConstructName','SubjectName']
x = df.drop(columns=[QuestionId_column] + variables_to_exclude)
y = df[QuestionId_column]

correlation_matrix = x.corr()
high_correlation_vars = correlation_matrix[abs(correlation_matrix) >0.9].stack().reset_index()
high_correlation_vars = high_correlation_vars[high_correlation_vars['level_0']!= high_correlation_vars['level_1']]
print("Highly correlated variables (multicolinearity):")
print(high_correlation_vars)

numerical_vars = x.select_dtypes(include=['float64','int64']).columns
categorical_vars = x.select_dtypes(include=['object','category']).columns

scaler = StandardScaler()
x[numerical_vars] = scaler.fit_transform(x[numerical_vars])

preprocessor = ColumnTransformer(
     transformers=[
         ('num', StandardScaler(), numerical_vars),
         ('cat', OneHotEncoder(), categorical_vars)
     ],
      remainder='drop')
x_processed = preprocessor.fit_transform(x)
x_train,x_test,y_train,y_test = train_test_split(x_processed,y,test_size=0.2,random_state=42)

print("Data preprocessing completed.Ready for model creation.")